Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Agent traces and sessions

**Tracks 1 and 3.** Requires Module 4 with the *Keep prompts and responses on the spans* patch applied
before `cdk deploy`.

You drive a three-turn conversation with a session ID you choose, confirm the spans landed in `aws/spans`,
learn the agent's OpenTelemetry service name, then read the trace tree in Omni.

In [ ]:
%pip install --quiet "boto3==1.43.126" "botocore==1.43.126"
import boto3

assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 43, 100), (
    f"boto3 too old: {boto3.__version__} (need >=1.43.100 for the cloudwatchomni client)"
)
# If the kernel imported an older botocore before the pip install above, the new
# service models are not loaded yet. Fail here with an actionable message.
services = boto3.session.Session().get_available_services()
assert "cloudwatchomni" in services and "observabilityadmin" in services, (
    "This kernel has no cloudwatchomni client. Restart the kernel (Kernel > Restart) and run this cell again."
)
print(f"boto3 {boto3.__version__} OK - cloudwatchomni and observabilityadmin clients available")

In [ ]:
import json
import os
import pathlib
import time

import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError("No AWS region configured. Set AWS_REGION or run `aws configure set region <region>`.")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# The CLI pages save identifiers as `export KEY=VALUE` lines in this file, so a
# participant can switch between the CLI and notebook paths at any step.
STATE_FILE = pathlib.Path.home() / ".workshop-omni.env"


def load_state():
    state = {}
    if STATE_FILE.exists():
        for line in STATE_FILE.read_text().splitlines():
            if line.startswith("export ") and "=" in line:
                key, value = line[len("export "):].split("=", 1)
                state[key] = value.strip()
    return state


def save_state(**values):
    state = load_state()
    state.update({k: v for k, v in values.items() if v})
    STATE_FILE.write_text("".join(f"export {k}={v}\n" for k, v in state.items()))
    return state


omni = boto3.client("cloudwatchomni", region_name=REGION)
obs_admin = boto3.client("observabilityadmin", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)
print(f"Region {REGION}, account ...{ACCOUNT_ID[-4:]}")

## Step 1: Invocation helpers

The same path as Test 5 on the Module 4 *Run the agent* page: a throwaway Cognito user, an access token,
an HTTPS POST to the runtime with the session ID in the header and the body.

In [ ]:
import secrets
import urllib.parse

import requests

ssm = boto3.client("ssm", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)


def fast_param(name):
    return ssm.get_parameter(Name=f"/FAST-stack/{name}")["Parameter"]["Value"]


RUNTIME_ARN = fast_param("runtime-arn")
USER_POOL_ID = fast_param("cognito-user-pool-id")
CLIENT_ID = fast_param("cognito-user-pool-client-id")
INVOKE_URL = (f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/"
              f"{urllib.parse.quote(RUNTIME_ARN, safe='')}/invocations?qualifier=DEFAULT")


def access_token_for_throwaway_user():
    """Create a throwaway Cognito user, return (access_token, username).

    The Runtime's JWT authorizer matches the token's client_id claim, so this must
    be the AccessToken, not the IdToken. Delete the user with delete_throwaway_user.
    """
    username = f"omni-{secrets.token_hex(4)}@example.com"
    password = secrets.token_urlsafe(18) + "!Aa1"
    cognito.admin_create_user(
        UserPoolId=USER_POOL_ID, Username=username, MessageAction="SUPPRESS",
        UserAttributes=[{"Name": "email", "Value": username}, {"Name": "email_verified", "Value": "true"}],
    )
    cognito.admin_set_user_password(UserPoolId=USER_POOL_ID, Username=username, Password=password, Permanent=True)
    auth = cognito.initiate_auth(
        AuthFlow="USER_PASSWORD_AUTH", ClientId=CLIENT_ID,
        AuthParameters={"USERNAME": username, "PASSWORD": password},
    )
    return auth["AuthenticationResult"]["AccessToken"], username


def delete_throwaway_user(username):
    cognito.admin_delete_user(UserPoolId=USER_POOL_ID, Username=username)


def ask(token, session_id, prompt):
    """One turn against the runtime. The session ID must be at least 33 characters."""
    response = requests.post(
        INVOKE_URL,
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json",
                 "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id},
        json={"prompt": prompt, "runtimeSessionId": session_id},
        timeout=120,
    )
    print(f"HTTP {response.status_code}  <- {prompt[:70]}")
    return response.text

## Step 2: One session, three turns

In [ ]:
SESSION_ID = f"omni-{secrets.token_hex(16)}-session"
token, username = access_token_for_throwaway_user()
try:
    ask(token, SESSION_ID, "Find flights from SFO to TYO on 2026-09-15")
    ask(token, SESSION_ID, "Which of those is cheapest, and find me a hotel in Tokyo for 3 nights from that date")
    ask(token, SESSION_ID, "Compare the total cost of the two cheapest flight and hotel combinations")
finally:
    delete_throwaway_user(username)
save_state(OMNI_SESSION_ID=SESSION_ID)
print(f"Session: {SESSION_ID}")

## Step 3: Confirm the spans landed, and learn the service name

Spans reach the shared `aws/spans` log group within about a minute. The service name
(`<runtime-name>.<endpoint>`, for example `FAST_stack_FASTAgent.DEFAULT`) identifies the agent in Omni and
in the online evaluation you create next.

In [ ]:
import collections

query = (f'fields @timestamp, name, traceId, resource.attributes.service.name as service'
         f' | filter attributes.session.id = "{SESSION_ID}"'
         ' | filter ispresent(scope.name) and ispresent(spanId) | sort @timestamp asc | limit 200')
for attempt in range(6):
    qid = logs.start_query(logGroupNames=["aws/spans"], startTime=int(time.time()) - 1800,
                           endTime=int(time.time()) + 300, queryString=query)["queryId"]
    while (result := logs.get_query_results(queryId=qid))["status"] not in ("Complete", "Failed", "Cancelled", "Timeout"):
        time.sleep(3)
    spans = [{f["field"]: f["value"] for f in row} for row in result.get("results", [])]
    if spans:
        break
    print("no spans yet, waiting 20 s ...")
    time.sleep(20)

print(f"{len(spans)} spans in the session")
print("span names:", dict(collections.Counter(s.get("name", "?") for s in spans)))
traces = sorted({s["traceId"] for s in spans if s.get("traceId")})
print(f"{len(traces)} traces (one per turn): {[t[-8:] for t in traces]}")
services = {s["service"] for s in spans if s.get("service")}
print("service name:", services)
if not spans:
    raise RuntimeError("Still no spans after two minutes: the runtime was deployed without the Module 4 "
                       "span patch, so spans went to the agent's own log group. Re-apply the patch and cdk deploy.")
SERVICE_NAME = next(iter(services))
save_state(OMNI_SERVICE_NAME=SERVICE_NAME)

## Step 4: Read the trace in Omni

In your space choose **Agents**, then **Agent traces**, filter by your session ID and open the second
trace (the hotel search):

| Span | What it proves |
|------|----------------|
| `invoke_agent` root | One trace per turn; session and user IDs from `trace_attributes` in `travel_agent.py` |
| `chat` spans with `gen_ai.*` attributes and the LLM Gateway as server address | Every model call went through Module 2's gateway, with prompt and response text attached |
| `execute_tool` spans named after Gateway tools | Tool calls went through the AgentCore Gateway, with latency and status per tool |
| Memory spans | The session manager loaded and stored turns |
| Code Interpreter span (third trace) | The sandbox ran; usually the longest span |

**Sessions** shows the three traces as one conversation, **Topology** the agent, gateway and tools as a
graph, and **Agent dashboards** request volume, errors, latency and token usage.

Continue with `06-evaluate.ipynb`.